# EuroSAT Baseline CNN Kaggle Runner

This notebook runs full local-equivalent Baseline CNN training on Kaggle using the same configuration policy as local runs.

In [ ]:
import shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient

WORKING = Path('/kaggle/working')
REPO = WORKING / 'satellite-land-classification-dl'

if REPO.exists():
    shutil.rmtree(REPO)

token = UserSecretsClient().get_secret('GH_TOKEN')
clone_url = f'https://{token}@github.com/milanovicmilos/satellite-land-classification-dl.git'

!git clone {clone_url} {REPO.as_posix()}
!git -C {REPO.as_posix()} remote set-url origin https://github.com/milanovicmilos/satellite-land-classification-dl.git
%cd {REPO.as_posix()}
!git checkout feature/efficientnet-optimization

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -e .

In [ ]:
import json

DATASET_INPUT_ROOT = Path('/kaggle/input/datasets/apollo2506/eurosat-dataset/EuroSAT')
assert DATASET_INPUT_ROOT.exists(), f'Missing dataset path: {DATASET_INPUT_ROOT}'

CONFIG_BASELINE = REPO / 'configs' / 'baseline_cnn_full.json'

payload = json.loads(CONFIG_BASELINE.read_text(encoding='utf-8'))
payload['dataset_root'] = DATASET_INPUT_ROOT.as_posix()
CONFIG_BASELINE.write_text(json.dumps(payload, indent=2), encoding='utf-8')

print('Patched baseline dataset_root:', DATASET_INPUT_ROOT)

In [ ]:
!python run.py --prepare-dataset --config configs/baseline_cnn_full.json --defaults configs/experiment.defaults.json --splits-output /kaggle/working/artifacts/splits

In [ ]:
!python run.py --run-baseline --config configs/baseline_cnn_full.json --defaults configs/experiment.defaults.json --splits-output /kaggle/working/artifacts/splits --reports-output /kaggle/working/artifacts/reports/baseline_cnn_full.json --checkpoints-output /kaggle/working/checkpoints/baseline_cnn

In [ ]:
!zip -r /kaggle/working/baseline_results.zip /kaggle/working/artifacts /kaggle/working/checkpoints